# Show Electrocatalyst Models


## Goal of the notebook

Let us turn catalyst structures and reaction pathways into clear figures. Each view should help a reader see the geometry and the modeling choices behind the calculation.


## Description of the method

ASE lets us inspect structures and prepare them for POV-Ray. Shared camera, light, atom size, and texture settings give each image a consistent visual language. We will relax selected structures with TBLite before we render them.

Our examples include oxides, metal surfaces, M–N–C sites, and iron phthalocyanine. We will also place intermediates into associative and dissociative ORR pathways. As we build each figure, let us ask what a reader can learn from the geometry.


## What we will do

1. Let us choose rendering settings and inspect oxide and catalyst structures.
2. We will prepare adsorbate models, choose substrate constraints, and create images or animations.
3. We will arrange intermediates into associative and dissociative ORR pathways.


In [ ]:
# Install course dependencies only in Google Colab. Local runs use environment.yml.
!if python -c "import google.colab" >/dev/null 2>&1; then apt-get update -qq && apt-get install -y povray && pip install -q ase tblite apng pillow matplotlib; fi

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

from apng import APNG, PNG
from ase import Atom, Atoms
from ase.build import fcc111, graphene_nanoribbon
from ase.constraints import FixAtoms
from ase.io import read, write
from ase.io.pov import get_bondpairs
from ase.optimize import BFGS
from ase.visualize import view
from IPython.display import Image as IPyImage, display
from pathlib import Path
from PIL import Image, ImageOps
from tblite.ase import TBLite

In [ ]:
# Standard publication figure sizes in inches
plt.rc("figure", figsize  = (8.25 / 2.54, 8.25 / 2.54))
plt.rc("figure", dpi      = 100)
plt.rc("lines",  linewidth= 4)
plt.rc("font",   size     = 10)
plt.rc("font",   family   = "sans-serif")
plt.rc("axes",   titlesize= 10)
plt.rc("axes",   labelsize= 9)
plt.rc("xtick",  labelsize= 8)
plt.rc("ytick",  labelsize= 8)
plt.rc("legend", fontsize = 8)
plt.rc("figure", titlesize= 10)

pd.options.display.float_format = "{:.2f}".format

## Choose the animation view

Set `animation_mode` to `"static"` for a slide or `"animated"` to view the catalyst series move through its structures. The static view shows the first model.


In [ ]:
animation_mode = "static"  # Choose "animated" or "static".

## Set the rendering style

Let us choose one POV-Ray camera, lighting, texture, and resolution setup. The shared settings will give the structures a consistent appearance.


In [ ]:
povray_settings = {"display": False, "transparent": False, "camera_type": "orthographic", "canvas_width": 4096, "canvas_height": None, "image_plane": None, "camera_dist": 1000, "depth_cueing": False, "point_lights": [], "area_light": [(0, 0, 100), "White", 1000, 1000, 1, 1], "celllinewidth": 0.05,}

## Explore oxide catalysts




Oxides can support oxygen reduction reaction (ORR) catalysts. Here we load two structures from a crystal database and replace one surface atom with platinum. This creates a model of an isolated Pt site in an oxide surface, as reported in studies of low Pt loading. Let us inspect where the Pt atom sits and which atoms surround it.


In [ ]:
# Download once into the current folder.
!if [ ! -s Fe2O3.cif ]; then wget -q https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Oxides/Fe2O3.cif -O Fe2O3.cif; fi
if not Path('Fe2O3.cif').is_file() or Path('Fe2O3.cif').stat().st_size == 0:
    raise FileNotFoundError("Download failed: Fe2O3.cif")
Fe2O3 = read("Fe2O3.cif")
Fe2O3 = Fe2O3 * (3, 3, 1)
Fe2O3[32].symbol = "Pt"
write("Fe2O3.xyz", Fe2O3 * (3, 3, 1))

In [ ]:
view(Fe2O3, viewer="x3d")

In [ ]:
# Download once into the current folder.
!if [ ! -s MnO2.cif ]; then wget -q https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Oxides/MnO2.cif -O MnO2.cif; fi
if not Path('MnO2.cif').is_file() or Path('MnO2.cif').stat().st_size == 0:
    raise FileNotFoundError("Download failed: MnO2.cif")
MnO2 = read("MnO2.cif")
MnO2 = MnO2 * (2, 9, 1)
MnO2[77].symbol = "Pt"
write("MnO2.xyz", MnO2 * (3, 12, 3))

In [ ]:
view(MnO2, viewer="x3d")

## Build metal–nitrogen–carbon (M–N–C) catalysts

> **Pause and predict:** Which shape features should remain visible as the metal or coordinating atom changes?
>
> **What to look for:** Follow the animation. Ask whether a shape change comes from the structure or from the camera.


M–N–C materials can use a single transition metal coordinated by nitrogen atoms in a graphene like carbon matrix. The `mnc_model()` function builds a graphene nanoribbon, replaces selected carbon atoms with other elements, and places the metal at the center.

Let us create twelve models with Mn, Fe, Ni, or Co and with N, C, or S around the metal. POV-Ray will render each structure, then an animated PNG will place them in sequence. The [APNG-to-GIF tool](https://ezgif.com/apng-to-gif) can convert that file to GIF format.


In [ ]:
def build_mnc_model(metal_symbol, ligand_1_symbol, ligand_2_symbol, ligand_3_symbol, ligand_4_symbol):
    mnc_structure = graphene_nanoribbon(4, 2, type="armchair", saturated=False, vacuum=3.5)
    mnc_structure.append(Atom(metal_symbol, position=(mnc_structure[11].position + mnc_structure[12].position) / 2))
    mnc_structure[10].symbol = ligand_1_symbol
    mnc_structure[13].symbol = ligand_2_symbol
    mnc_structure[18].symbol = ligand_3_symbol
    mnc_structure[21].symbol = ligand_4_symbol
    del mnc_structure[[11, 12, 24, 27, 28, 31]]
    write(f"{metal_symbol}{ligand_1_symbol}{ligand_2_symbol}{ligand_3_symbol}{ligand_4_symbol}.xyz", mnc_structure)
    generic_projection_settings = {"rotation": "90x,0y,0z", "radii": 0.9, "colors": None, "show_unit_cell": 0,}
    textures = []
    for atom in mnc_structure:
        textures.append("jmol")
    write(f"{metal_symbol}{ligand_1_symbol}{ligand_2_symbol}{ligand_3_symbol}{ligand_4_symbol}.pov", mnc_structure, **generic_projection_settings, povray_settings=povray_settings)

In [ ]:
systems = [
    ["Mn", "N", "N", "N", "N"],
    ["Fe", "N", "N", "N", "N"],
    ["Ni", "N", "N", "N", "N"],
    ["Co", "N", "N", "N", "N"],
    ["Mn", "C", "C", "N", "N"],
    ["Fe", "C", "N", "C", "N"],
    ["Ni", "C", "N", "N", "C"],
    ["Co", "C", "C", "C", "C"],
    ["Mn", "S", "N", "N", "N"],
    ["Fe", "N", "S", "N", "N"],
    ["Ni", "N", "N", "S", "N"],
    ["Co", "N", "N", "N", "S"],
]
for elements in systems:
    metal_symbol = elements[0]
    ligand_1_symbol = elements[1]
    ligand_2_symbol = elements[2]
    ligand_3_symbol = elements[3]
    ligand_4_symbol = elements[4]
    build_mnc_model(metal_symbol, ligand_1_symbol, ligand_2_symbol, ligand_3_symbol, ligand_4_symbol)
    name = f"{metal_symbol}{ligand_1_symbol}{ligand_2_symbol}{ligand_3_symbol}{ligand_4_symbol}"
    os.system(f"povray +I{name}.pov +O{name}.png +A +AM2 +UA +Q9 -D")

In [ ]:
scaling_factor = 7 / 10
png_files = []
for elements in systems:
    png_files.append(f"{''.join(elements)}.png")

In [ ]:
png_files

In [ ]:
for file in png_files:
    if os.path.exists(file):
        with Image.open(file) as image:
            new_width = int(image.width * scaling_factor)
            new_height = image.height
            resized_img = image.resize((new_width, new_height), Image.LANCZOS)
            resized_img.save(file)
print("All specified PNG files have been resized.")

In [ ]:
png_files

In [ ]:
if animation_mode == "animated":
    apng = APNG()
    for file in png_files:
        apng.append(PNG.open(file), delay=500)
    apng.save("model_MNC_animated.png")
    display(IPyImage(filename="model_MNC_animated.png"))
else:
    display(Image.open(png_files[0]))

The display choice at the start of this notebook lets you view one model as a still image or the full M–N–C series as an animation.


## Show an OH–H₂O adlayer on Pt(111)




Here we load a relaxed Pt(111) slab with OH and water molecules on its surface. Let us inspect it in the interactive viewer, then render a static image with POV-Ray. Change the rotation in `generic_projection_settings` to choose a new view. You can also adjust the texture setting.


In [ ]:
# Download once into the current folder.
!if [ ! -s Pt-OH-H2O.traj ]; then wget -q https://github.com/double-layer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj -O Pt-OH-H2O.traj; fi
if not Path('Pt-OH-H2O.traj').is_file() or Path('Pt-OH-H2O.traj').stat().st_size == 0:
    raise FileNotFoundError("Download failed: Pt-OH-H2O.traj")
Pt111 = read("Pt-OH-H2O.traj", -1)
view(Pt111, viewer="x3d")

In [ ]:
generic_projection_settings = {"rotation": "225x,215.264y,30z", "radii": 1.1, "colors": None, "show_unit_cell": 2,}

povray_settings["camera_type"] = "orthographic"
atom_textures = []
for atom in Pt111:
    atom_textures.append("intermediate")
povray_settings["textures"] = atom_textures
write(f"Pt111.pov", Pt111, **generic_projection_settings, povray_settings=povray_settings)
os.system(f"povray +IPt111.pov +Omodel_Pt111.png +A +AM2 +UA +Q11 +W1024 +H3072 -D")

## Place OH on an M–N–C catalyst




Let us use `mnc_model()` to build an FeN₄ site. We will place OH above Fe and relax the adsorbate with BFGS and GFN1-xTB. `FixAtoms` holds the carbon framework in place. Then we will render the relaxed structure with POV-Ray.


In [ ]:
# Build the FeN4 model directly for this adsorption example.
fe_n4_structure = graphene_nanoribbon(4, 2, type="armchair", saturated=False, vacuum=3.5)
fe_n4_structure.append(Atom('Fe', position=(fe_n4_structure[11].position + fe_n4_structure[12].position) / 2))
fe_n4_structure[10].symbol = 'N'
fe_n4_structure[13].symbol = 'N'
fe_n4_structure[18].symbol = 'N'
fe_n4_structure[21].symbol = 'N'
del fe_n4_structure[[11, 12, 24, 27, 28, 31]]

### Find the atoms in FeN₄

Find Fe in the atom list before you use its index. Python starts its index count at zero.


In [ ]:
print("Formula:", fe_n4_structure.get_chemical_formula())
display(pd.DataFrame({"index": range(len(fe_n4_structure)), "element": fe_n4_structure.get_chemical_symbols()}).tail(6))

In [ ]:
fe_n4_structure.center(vacuum=6.0, axis=1)
for atom in fe_n4_structure:
    if atom.symbol == "Fe":
        fe_index = atom.index
        break
fe_n4_structure.set_constraint(FixAtoms(indices=list(range(len(fe_n4_structure)))))
fe_position_angstrom = fe_n4_structure[fe_index].position
o_position_angstrom = fe_position_angstrom + np.array([0.0, 1.8, 0.0])
h_position_angstrom = o_position_angstrom + np.array([0.5, 0.5, 0.0])
fe_n4_structure += Atoms("OH", positions=[o_position_angstrom, h_position_angstrom])
fe_n4_structure.calc = TBLite(method="GFN1-xTB")
optimizer = BFGS(fe_n4_structure, trajectory="FeN4.traj", logfile="FeN4.log")
optimizer.run(fmax=0.2)

### Check the FeN₄ adsorption geometry

Inspect the optimized structure and the energy shown above. Which change in geometry or energy matters for the chemical question in this section?

In [ ]:
view(fe_n4_structure, viewer="x3d")

In [ ]:
generic_projection_settings = {"rotation": "225x,215.264y,30z", "radii": 1.1, "colors": None, "show_unit_cell": 2,}
atom_textures = []
for atom in fe_n4_structure:
    atom_textures.append("intermediate")
povray_settings["textures"] = atom_textures
write(f"FeN4.pov", fe_n4_structure, **generic_projection_settings, povray_settings=povray_settings)
os.system(f"povray +IFeN4.pov +Omodel_FeN4.png +A +AM2 +UA +Q11 +W1024 +H3072 -D")

## Place OH on iron phthalocyanine




Iron phthalocyanine (FePc) has a planar Fe–N₄ site in a discrete molecule. M–N–C has a similar local arrangement in a carbon framework. Here we fetch the FePc structure, place OH above Fe, and relax the adsorbate as we did for FeN₄. Let us look at the four coordinating nitrogen atoms, then notice how the ligand environment and full geometry differ.


In [ ]:
# Download once into the current folder.
!if [ ! -s FePc.xyz ]; then wget -q https://gitlab.com/doublelayer/test-models/-/raw/main/iron-phthalocyanine-FePc-OH/FePc.xyz -O FePc.xyz; fi
if not Path('FePc.xyz').is_file() or Path('FePc.xyz').stat().st_size == 0:
    raise FileNotFoundError("Download failed: FePc.xyz")
fe_pc_structure = read("FePc.xyz")
for atom in fe_pc_structure:
    if atom.symbol == "Fe":
        fe_index = atom.index
        break
fe_pc_structure.set_constraint(FixAtoms(indices=list(range(len(fe_pc_structure)))))
fe_position_angstrom = fe_pc_structure[fe_index].position
o_position_angstrom = fe_position_angstrom + np.array([0.0, 0.0, 1.8])
h_position_angstrom = o_position_angstrom + np.array([0.5, 0.0, 0.5])
fe_pc_structure += Atoms("OH", positions=[o_position_angstrom, h_position_angstrom])
fe_pc_structure.calc = TBLite(method="GFN1-xTB")
optimizer = BFGS(fe_pc_structure, trajectory="FePc.traj", logfile="FePc.log")
optimizer.run(fmax=0.2)

### Check the FePc adsorption geometry


In [ ]:
view(fe_pc_structure, viewer="x3d")

In [ ]:
generic_projection_settings = {"rotation": "225x,215.264y,30z", "radii": 1.1, "colors": None, "show_unit_cell": 0,}
atom_textures = []
for atom in fe_pc_structure:
    atom_textures.append("intermediate")
povray_settings["textures"] = atom_textures
write(f"FePc.pov", fe_pc_structure, **generic_projection_settings, povray_settings=povray_settings)
os.system(f"povray +IFePc.pov +Omodel_FePc.png +A +AM2 +UA +Q11 +W2048 +H1024 -D")

This section arranges the associative and dissociative ORR pathways on Pt(111). We place each intermediate to make the sequence clear in a figure. The combined model is a schematic for rendering, so we use its placed coordinates directly.

> **Before you begin:** Choose the intermediates that you want the figure to show.
>
> **What to look for:** Follow both pathways across the Pt surface.


### Build the slab


To model ORR on Pt(111), we will build a periodic slab with `fcc111`. The parameters `a` and `b` set the number of unit cells along x and y. The parameter `c` sets the number of layers, and `v` sets the vacuum thickness. `FixAtoms` holds the Pt atoms in place while the adsorbates relax. We will also calculate the nearest-neighbor Pt–Pt distance, $M_M$, as a guide for adsorbate positions.


In [ ]:
surface_cells_x = 16
surface_cells_y = 4
surface_layers = 1
vacuum_angstrom = 8

In [ ]:
slab = fcc111("Pt", size=(surface_cells_x, surface_cells_y, surface_layers), vacuum=vacuum_angstrom, orthogonal=True)

In [ ]:
# [:, column] selects that column for all rows of a NumPy array.
slab.set_constraint(FixAtoms(mask=slab.get_tags() > 0))
slab.center()
slab.positions[:, 2] += -vacuum_angstrom / 2

In [ ]:
distances = slab.get_all_distances()
pt_pt_distance_angstrom = np.min(distances[np.nonzero(distances)])

### Build the associative pathway


In the associative ORR pathway, O₂ keeps its O–O bond as the reaction proceeds. We place O₂, OOH, O, and OH on one Pt(111) slab. These coordinates prepare a clear pathway illustration.


In [ ]:
# .copy() creates an independent object so changes do not overwrite the original.
assoc = slab.copy()
o2 = Atoms("O2", positions=[(0, 0, 0), (-0.7, -1, 0)])
o2.positions += [2 * pt_pt_distance_angstrom, np.sqrt(3) * pt_pt_distance_angstrom, 1.5,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
assoc += o2
ooh = Atoms("OOH", positions=[(0, 0, 0), (0, 0, 1.4), (0.77, 0, 2.1)])
ooh.positions += [5 * pt_pt_distance_angstrom, np.sqrt(3) * pt_pt_distance_angstrom, 1.7,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
assoc += ooh
o = Atoms("O", positions=[(0, 0, 0)])
o.positions += [7.5 * pt_pt_distance_angstrom, np.sqrt(3) * pt_pt_distance_angstrom - 0.3 * pt_pt_distance_angstrom, 1.6,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
assoc += o
oh = Atoms("OH", positions=[(0, 0, 0), (0.5, -0.4, 0.96)])

In [ ]:
oh.positions += [11 * pt_pt_distance_angstrom, np.sqrt(3) * pt_pt_distance_angstrom, 1.7,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
assoc += oh

In [ ]:
assoc.calc = TBLite(method="GFN1-xTB", accuracy=1, max_iterations=500)
optimizer = BFGS(assoc, trajectory="assoc.traj", logfile="assoc.log")
# Deliberately loose teaching threshold; this does not establish an accurate geometry.
optimizer.run(fmax=5)

### Check the associative model


In [ ]:
assoc = read("assoc.traj", -1)
view(assoc, viewer="x3d")

### Build the dissociative pathway


In the dissociative pathway, O₂ splits into two O atoms on adsorption. We place O+O, OOH, OH+OH, and H₂O on the same Pt(111) model to show this sequence.


In [ ]:
disso = slab.copy()

In [ ]:
o2_ad = Atoms("O2", positions=[(0, 0, 0), (pt_pt_distance_angstrom, 0, 0)])
o2_ad.positions += [2.5 * pt_pt_distance_angstrom, np.sqrt(3) * pt_pt_distance_angstrom - 0.3 * pt_pt_distance_angstrom, 1.5,] + disso[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
disso += o2_ad
ooh = Atoms("OOH", positions=[(0, 0, 0), (pt_pt_distance_angstrom, 0, 0), (0.77, 0, 0.8)])
ooh.positions += [6 * pt_pt_distance_angstrom, np.sqrt(3) * pt_pt_distance_angstrom, 1.7,] + disso[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
disso += ooh
o2h2 = Atoms(
    "O2H2",
    positions=[
        (0, 0, 0),
        (pt_pt_distance_angstrom, 0, 0),
        (0.5, -0.4, 0.96),
        (pt_pt_distance_angstrom + 0.5, -0.4, 0.96),
    ])

In [ ]:
o2h2.positions += [9 * pt_pt_distance_angstrom, np.sqrt(3) * pt_pt_distance_angstrom, 1.7,] + disso[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
disso += o2h2
h2o = Atoms("OH2", positions=[(0, 0, 0), (0.5, -0.2, 0.7), (-0.5, -0.2, 0.7)])
h2o.positions += [13 * pt_pt_distance_angstrom, np.sqrt(3) * pt_pt_distance_angstrom, 1.9,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
disso += h2o

In [ ]:
disso.calc = TBLite(method="GFN1-xTB", accuracy=1, max_iterations=500)
optimizer = BFGS(disso, trajectory="disso.traj", logfile="disso.log")
# Deliberately loose teaching threshold; this does not establish an accurate geometry.
optimizer.run(fmax=5)

### Check the dissociative model


In [ ]:
disso = read("disso.traj", -1)
view(disso, viewer="x3d")

### Place both pathways on one slab


Let us place the associative and dissociative models side by side on one extended slab. We will also add gas phase O₂ and H₂O above the surface. This gives us one view of the ORR steps on Pt(111).


In [ ]:
model = assoc.copy()
back = disso.copy()
back.positions[:, 1] += 2 * np.sqrt(3) * pt_pt_distance_angstrom
model += back
cell = model.get_cell()
cell[1, 1] *= 2
model.set_cell(cell)
model.center()
model.rotate(180, "z")
model.wrap()
h2o = Atoms("OH2", positions=[(0, 0, 0), (-0.4, 0.4, 0.7), (0.4, -0.4, 0.7)])
h2o.positions += [-2 * pt_pt_distance_angstrom, 5 * np.sqrt(3) / 2 * pt_pt_distance_angstrom, 5,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1) + 1].position
model += h2o
o2 = Atoms("O2", positions=[(0, 0, 0), (0, 0, 1.23)])
o2.positions += [16.5 * pt_pt_distance_angstrom, 1 * pt_pt_distance_angstrom, 5,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
model += o2
h2o = Atoms("OH2", positions=[(0, 0, 0), (-0.4, -0.4, 0.7), (0.4, 0.4, 0.7)])

In [ ]:
h2o.positions += [5 * pt_pt_distance_angstrom, 8 * np.sqrt(3) / 2 * pt_pt_distance_angstrom, 5,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
model += h2o
h2o = Atoms("OH2", positions=[(0, 0, 0), (-0.4, -0.4, 0.7), (0.4, 0.4, 0.7)])
h2o.positions += [8.5 * pt_pt_distance_angstrom, 8 * np.sqrt(3) / 2 * pt_pt_distance_angstrom, 5,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
model += h2o
h2o2 = Atoms("O2H2", positions=[(0, 0, 0), (1.23, 0, 0), (-0.9, 0, 0), (1.23, 0, 0.9)])
h2o2.positions += [12 * pt_pt_distance_angstrom, 8 * np.sqrt(3) / 2 * pt_pt_distance_angstrom, 5,] + assoc[surface_cells_x * surface_cells_y * (surface_layers - 1)].position
model += h2o2
model.pbc = [False, False, False]
view(model, viewer="x3d")

In [ ]:
# This combined pathway is a schematic for rendering; its geometry is not used for an energy calculation.

### Render the combined model


Here are the POV-Ray settings for the combined model. The rotation gives us a view from above, with both pathways in sight. Render the figure and inspect the position of each intermediate.


In [ ]:
view(model, viewer="x3d")

In [ ]:
# Use a slightly expanded covalent-radius cutoff to show stretched bonds.
bondpairs = get_bondpairs(model, radius=1.1)
generic_projection_settings = {"rotation": "-45x, 0y, 0z", "radii": 1.1, "colors": None, "show_unit_cell": 0,}
atom_textures = []
for atom in model:
    atom_textures.append("intermediate")
povray_settings["textures"] = atom_textures
write(f"model.pov", model, **generic_projection_settings, povray_settings=povray_settings)
os.system(f"povray +Imodel.pov +Omodel.png +A +AM2 +UA +Q11 +W4096 +H2048 -D")

In [ ]:
image = Image.open("model.png")
ratio = 1.5
resized_image = image.resize((int(image.width * ratio), int(image.height)))
mirrored_image = ImageOps.mirror(resized_image)
mirrored_image.save("model_fix.png")

In [ ]:
mirrored_image

## Student exercise: Show an adsorption geometry

Let us choose a catalyst model and prepare a view of its active site, OH group, and support. Identify the atoms that the calculation holds fixed. Explain what the geometry tells us, then name one claim about activity that this image alone cannot support.

Record a prediction, one numerical or graphical result, and your chemical interpretation. Copy a structure or value so the main calculation stays intact.


In [ ]:
# Your code here.
# Use a separate variable for your changed parameter and display the comparison.

### Your answer

Prediction: [write your prediction]

Result: [report a value or refer to your figure]

Chemical explanation: [explain the result]

Assumption or limitation: [state one limitation]


## What have you learned

Let us gather the main ideas from this notebook. You can now:

- Build and inspect surface and molecular catalyst models.
- Choose which atoms to constrain during adsorbate relaxation.
- Create static images and animations from ASE structures.
- Show adsorption sites, bonds, and pathway details clearly in a figure.


### Questions for discussion

1. Which parts of an active site should stay in view as we choose the camera angle?
2. Which calculation could support a claim that one illustrated pathway is favored?


The display choice at the start of this notebook lets you view one model as a still image or the full M–N–C series as an animation.
